In [ ]:
!pip install transformers datasets evaluate accelerate
!pip install -q peft "torchao>=0.16.0"
!pip install minigrid
!pip install stable-baselines3[extra]

In [ ]:
import minigrid
from minigrid.wrappers import ImgObsWrapper
from stable_baselines3 import PPO
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
import gymnasium as gym
from minigrid.wrappers import ImgObsWrapper
import torch as th
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt
import os
from stable_baselines3.common.evaluation import evaluate_policy
import numpy as np
import seaborn as sns
from stable_baselines3.common.monitor import Monitor
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
import evaluate
import re
import shutil
import time
from google.colab import files
from peft import LoraConfig, get_peft_model, TaskType

In [ ]:
class MinigridFeaturesExtractor(BaseFeaturesExtractor):
    def __init__(self, observation_space, features_dim=128):
        super().__init__(observation_space, features_dim)
        n_input_channels = observation_space.shape[2]

        self.cnn = nn.Sequential(
            nn.Conv2d(n_input_channels, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Flatten(),
        )

        with th.no_grad():
            sample_tensor = th.as_tensor(observation_space.sample()[None]).float()
            permuted_sample = sample_tensor.permute(0, 3, 1, 2)
            n_flatten = self.cnn(permuted_sample).shape[1]

        self.linear = nn.Sequential(nn.Linear(n_flatten, features_dim), nn.ReLU())

    def forward(self, observations: th.Tensor) -> th.Tensor:
        permuted_observations = observations.permute(0, 3, 1, 2)
        return self.linear(self.cnn(permuted_observations))

In [ ]:
import gymnasium as gym
import numpy as np

class IntrinsicMotivationWrapper(gym.Wrapper):
    """
    A count-based exploration wrapper that provides intrinsic rewards
    for visiting novel (x, y) positions in the MiniGrid environment.
    """
    def __init__(self, env, beta=0.01):
        super().__init__(env)
        self.beta = beta
        self.visit_counts = {}

    def reset(self, **kwargs):
        return self.env.reset(**kwargs)

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        agent_pos = tuple(self.unwrapped.agent_pos)

        if agent_pos not in self.visit_counts:
            self.visit_counts[agent_pos] = 0
        self.visit_counts[agent_pos] += 1

        # Calculate intrinsic reward: beta / sqrt(N(s))
        intrinsic_reward = self.beta / np.sqrt(self.visit_counts[agent_pos])

        # Combine environment reward with intrinsic motivation bonus
        total_reward = reward + intrinsic_reward

        return obs, total_reward, terminated, truncated, info

print("IntrinsicMotivationWrapper defined!")

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from minigrid.wrappers import ImgObsWrapper
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor

im_log_dir = "./im_logs/"
os.makedirs(im_log_dir, exist_ok=True)

env_id = "MiniGrid-DoorKey-8x8-v0"
im_env = gym.make(env_id, max_episode_steps=200)

im_env = IntrinsicMotivationWrapper(im_env, beta=0.05)
im_env = ImgObsWrapper(im_env)
im_env = Monitor(im_env, im_log_dir)

policy_kwargs = dict(
    features_extractor_class=MinigridFeaturesExtractor,
    features_extractor_kwargs=dict(features_dim=128),
)

print("Starting PPO with Intrinsic Motivation training")
im_model = PPO(
    "CnnPolicy",
    im_env,
    policy_kwargs=policy_kwargs,
    verbose=1,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=256,
    n_epochs=10
)

im_model.learn(total_timesteps=1_000_000)
im_model.save("im_agent_doorkey")
print("\n Intrinsic Motivation Agent trained and saved!")
im_env.close()

im_log_file = os.path.join(im_log_dir, "monitor.csv")

if os.path.exists(im_log_file):
    print("Plotting training curves...")
    df_im = pd.read_csv(im_log_file, skiprows=1)

    rolling_window = 50

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    if len(df_im) >= rolling_window:
        axes[0].plot(df_im['l'].cumsum(), df_im['r'].rolling(window=rolling_window).mean(), color='blue', label='Intrinsic Motivation')

    axes[0].set_title(f'IM: Episode Rewards ({rolling_window}-Ep Rolling Mean)')
    axes[0].set_xlabel('Timesteps')
    axes[0].set_ylabel('Reward')
    axes[0].legend()
    axes[0].grid(True)

    if len(df_im) >= rolling_window:
        axes[1].plot(df_im['l'].cumsum(), df_im['l'].rolling(window=rolling_window).mean(), color='blue', label='Intrinsic Motivation')

    axes[1].set_title(f'IM: Episode Lengths ({rolling_window}-Ep Rolling Mean)')
    axes[1].set_xlabel('Timesteps')
    axes[1].set_ylabel('Length')
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.show()
else:
    print("Could not find log files to plot.")